## STEP 1 Knowledge Graph 기초

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
client = OpenAI()
df = pd.read_excel("Articles_20260312_215553.xlsx")

text = df.iloc[0]['content']

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 형태의 트리플을 추출하세요.  각 줄에 하나씩, 형식: (주어, 관계, 목적어)
  최대 10개만 추출하세요."""},
        {"role": "user", "content": text}
    ]
)

print("=== 추출된 트리플 ===")
print(response.choices[0].message.content)

=== 추출된 트리플 ===
1. (이재명 대통령, 강조하다, 신속한 추경의 필요성)
2. (이재명 대통령, 주재하다, 수석 보좌관 회의)
3. (이재명 대통령, 말하다, 추경 편성을 안 할 수 없다)
4. (이재명 대통령, 당부하다, 시간은 최소한으로 줄이고)
5. (이재명 대통령, 지적하다, 민생과 경제 산업 전반에 우려가 커지고 있다)
6. (이재명 대통령, 말하다, 소비, 투자 심리가 위축될 수 있다)
7. (이재명 대통령, 강조하다, 민생경제 충격 완화를 위한 골든타임을 허비해서는 안 되겠다)
8. (이재명 대통령, 밝혀다, 유류세 인하와 화물차·대중교통·농업인 유가 보조금 지원에 속도를 내야 되겠다)
9. (이재명 대통령, 말하다, 직접 지원으로 바꾸고 차등 지원이 필요하다)
10. (이재명 대통령, 평가하다, 국민 물가 부담 완화와 민생 안정에 큰 도움이 될 것)


In [5]:
all_triples = []

for idx in range(5):  # 기사 5개만
    text = df.iloc[idx]['content']
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 트리플을 추출하세요.
  각 줄에: (주어, 관계, 목적어)
  최대 5개만."""},
            {"role": "user", "content": text}
        ]
    )

    triples_text = response.choices[0].message.content
    print(f"\n--- 기사 {idx}: {df.iloc[idx]['title'][:30]} ---")
    print(triples_text)

    # 파싱
    for line in triples_text.strip().split("\n"):
        line = line.strip().strip("()")
        parts = [p.strip().strip("()") for p in line.split(",")]
        if len(parts) == 3:
            all_triples.append(tuple(parts))

print(f"\n총 트리플 수: {len(all_triples)}")


--- 기사 0: 이 대통령 “추경 편성 최대한 신속하게…상반기 공공요금 ---
1. (이재명 대통령, 강조하다, 추경의 필요성)
2. (이재명 대통령, 주재하다, 수석 보좌관 회의)
3. (이재명 대통령, 지적하다, 국제 에너지 수급 불안정성)
4. (이재명 대통령, 말하다, 민생경제 충격 완화의 중요성)
5. (이재명 대통령, 부탁하다, 정책 판단)

--- 기사 1: 李대통령, 다음주 ‘자본시장 정상화’ 간담회… 체질 개 ---
1. (이재명 대통령, 만나다, 상장 기업과 투자자들)  
2. (이재명 대통령, 주재하다, 수석보좌관회의)  
3. (이 대통령, 주재하다, 간담회)  
4. (강유정 대변인, 전하다, 간담회 내용)  
5. (간담회, 논의하다, 자본시장 안정화 방안과 개혁 과제들)

--- 기사 2: 한국법조인협회, 사법시험 부활 논의 중단 촉구 ---
1. (한국법조인협회, 촉구하다, 사법시험 부활 논의 중단)
2. (한법협, 밝히다, 입장)
3. (청와대 측, 나서다, 진화)
4. (사법시험, 양산하다, 고시 낭인)
5. (한법협, 주장하다, 로스쿨 도입 이후 변호사시험 합격자 수)

--- 기사 3: ‘사노맹’ 출신 백태웅 교수, 주OECD 대사에 임명… ---
1. (백태웅, 하와이대 로스쿨 교수, 윤석열 탄핵 정국과 관련해 인터뷰하고 있다)
2. (백태웅, 주OECD 한국대사에 임명되다, 외교부)
3. (백 신임 대사, 서울대 법대에 입학하다, 1981년)
4. (백 대사, 학생운동을 이끌다, 학도호국단 총학생회장)
5. (백태웅, 하와이대 로스쿨에서 국제인권법 등을 가르치다, 하와이대)

--- 기사 4: 李대통령, 대미투자법 통과에 "국가 과제 앞에 여야 따 ---
1. (이재명 대통령, 발언, "국가적 과제 앞에서 여야가 따로 없다는 것을 보여준 뜻깊은 사례")
2. (이재명 대통령, 감사를 전하다, 국회)
3. (특별법, 통과하다, 국회 본회의)
4. (정부, 추진하다, 특별법 시행을 위한 준비와 후속 조치)
5. (한미전략

In [6]:
from collections import Counter

# 모든 주어와 목적어 수집
entities = []
for s, r, o in all_triples:
    entities.append(s)
    entities.append(o)

counter = Counter(entities)
print("=== 자주 등장하는 엔티티 ===")
for entity, count in counter.most_common(10):
    print(f"  {entity}: {count}회")

=== 자주 등장하는 엔티티 ===
  1. (이재명 대통령: 3회
  2. (이재명 대통령: 3회
  추경의 필요성: 1회
  수석 보좌관 회의: 1회
  3. (이재명 대통령: 1회
  국제 에너지 수급 불안정성: 1회
  4. (이재명 대통령: 1회
  민생경제 충격 완화의 중요성: 1회
  5. (이재명 대통령: 1회
  정책 판단: 1회


## STEP 2 Neo4j & Cypher 쿼리

```shell
docker run -d \
  --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password123 \
  neo4j:latest
```

In [7]:
# uv add neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

with driver.session() as session:
    result = session.run("RETURN 'Neo4j 연결 성공!' AS message")
    print(result.single()["message"])

Neo4j 연결 성공!


In [8]:
  with driver.session() as session:
    # 노드 생성
    session.run("CREATE (t:Person {name: $name, role: $role})", name="트럼프", role="대통령")
    session.run("CREATE (c:Country {name: $name})", name="미국")
    session.run("CREATE (c:Country {name: $name})", name="이란")

    # 관계 생성
    session.run("""
          MATCH (p:Person {name: "트럼프"}), (c:Country {name: "미국"})
          CREATE (p)-[:대통령]->(c)
      """)
    session.run("""
          MATCH (a:Country {name: "미국"}), (b:Country {name: "이란"})
          CREATE (a)-[:전쟁중]->(b)
      """)

    # 검색
    result = session.run("""
          MATCH (p:Person)-[:대통령]->(c:Country)-[:전쟁중]->(target)
          RETURN p.name AS person, target.name AS target
      """)
    for record in result:
        print(f"{record['person']}은 {record['target']}과 전쟁 중인 나라의 대통령")

트럼프은 이란과 전쟁 중인 나라의 대통령


## STEP 3 데이터로 그래프 만들기

In [10]:
from openai import OpenAI
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import json

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

df = pd.read_excel("Articles_20260312_215553.xlsx")

def extract_triples(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """뉴스 기사에서 엔티티와 관계를 추출하세요.
  JSON 배열로 출력. 각 항목: {"subject": "주어", "subject_type": "타입", "relation": "관계", "object": "목적어",
  "object_type": "타입"}
  타입은: Person, Organization, Country, Event, Weapon, Place 중 하나.
  최대 7개만. JSON만 출력하세요. 마크다운 코드블록 쓰지 마세요."""},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"}
    )
    try:
        result = json.loads(response.choices[0].message.content)
        # {"triples": [...]} 형태일 수도 있으니 처리
        if isinstance(result, dict):
            for key in result:
                if isinstance(result[key], list):
                    return result[key]
        if isinstance(result, list):
            return result
        return []
    except:
        return []

# 테스트
triples = extract_triples(df.iloc[0]['content'])
for t in triples:
    print(f"({t['subject']}) -[{t['relation']}]-> ({t['object']})")


(이재명 대통령) -[강조]-> (신속한 추경의 필요성)
(이재명 대통령) -[주재]-> (청와대에서 수석 보좌관 회의)
(이재명 대통령) -[지적]-> (상반기 공공요금 동결, 농축수산물 할인)
(중동 상황) -[영향 미침]-> (국제 에너지 수급 불안정성)
(소상공인) -[매출 증대]-> (지역화폐로 지원)
(이재명 대통령) -[감사 인사]-> (위기 극복에 동참한 기업들)
(부처들) -[주문]-> (조사와 추적, 시정 조치)


In [11]:
# 기존 데이터 정리
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

all_triples = []

for idx, row in df.iterrows():
    if not isinstance(row['content'], str):
        continue

    triples = extract_triples(row['content'])
    print(f"[{idx+1}/{len(df)}] {row['title'][:30]}... → {len(triples)}개 트리플")

    for t in triples:
        t['source_article'] = row['title']
        t['category'] = row['category']
        all_triples.append(t)

print(f"\n총 트리플 수: {len(all_triples)}")

[1/60] 이 대통령 “추경 편성 최대한 신속하게…상반기 공공요금... → 7개 트리플
[2/60] 李대통령, 다음주 ‘자본시장 정상화’ 간담회… 체질 개... → 7개 트리플
[3/60] 한국법조인협회, 사법시험 부활 논의 중단 촉구... → 7개 트리플
[4/60] ‘사노맹’ 출신 백태웅 교수, 주OECD 대사에 임명…... → 7개 트리플
[5/60] 李대통령, 대미투자법 통과에 "국가 과제 앞에 여야 따... → 7개 트리플
[6/60] 범여권 의원 13명 "조희대 탄핵소추 추진할 것"... → 7개 트리플
[7/60] 이준석 “與 분열 기댄 낙관론은 ‘독’...‘공천 취소... → 7개 트리플
[8/60] 합참의장, 연합사령관과 백령도 해병부대 방문…작전태세 ... → 7개 트리플
[9/60] '공소취소 거래설' 파장 확산 … 정청래 "강력대응 나... → 7개 트리플
[10/60] 추미애, 경기지사 출마 공식선언‥"당당한 경기도 만들 ... → 7개 트리플
[11/60] 황건일 금통위원 "특정방향 기대 형성보단 당분간 신중한... → 7개 트리플
[12/60] 라면·식용유업계 우르르 가격 내렸다…"물가 부담 완화 ... → 7개 트리플
[13/60] 이한우 현대건설 대표 "원전기술로 북유럽 에너지 전환 ... → 7개 트리플
[14/60] 박홍근 "재정이 성장의 견인차 역할해야…정책적 역량 집... → 7개 트리플
[15/60] 석유 '최고가격제' 내일부터…휘발유 도매가 1724원으... → 7개 트리플
[16/60] LS, 12개 계열사 작년 영업익 1.5조…역대 최대 ... → 7개 트리플
[17/60] 삼성생명 “유배당 보험 역마진...삼전 지분 매각해도 ... → 7개 트리플
[18/60] 강남3구 아파트값 하락폭 커져...서울 상승세 6주째 ... → 7개 트리플
[19/60] “소프트웨어 기업엔 돈 못 대겠다”…세계 최대 투자은행... → 7개 트리플
[20/60] 우리은행 이사회에 금융소비자보호위원회 신설... → 6개 트리플
[21/

In [12]:
import re

def clean_rel_name(name):
    """Neo4j 관계 타입에 쓸 수 있도록 정제"""
    name = name.replace(" ", "_")
    name = re.sub(r'[^a-zA-Z0-9가-힣_]', '', name)  # 허용 문자만 남김
    if name and name[0].isdigit():
        name = "R_" + name  # 숫자로 시작하면 접두사 추가
    return name or "RELATED"

with driver.session() as session:
    for t in all_triples:
        try:
            rel_type = clean_rel_name(t['relation'])
            session.run("""
                  MERGE (s {name: $subject})
                  SET s:""" + t['subject_type'] + """
                  MERGE (o {name: $object})
                  SET o:""" + t['object_type'] + """
                  MERGE (s)-[r:""" + rel_type + """]->(o)
              """, subject=t['subject'], object=t['object'])
        except Exception as e:
            print(f"오류: {e}")
            continue

print("Neo4j 저장 완료!")

Neo4j 저장 완료!


In [13]:
with driver.session() as session:
    # 전체 노드/관계 수
    nodes = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    rels = session.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]
    print(f"노드: {nodes}개, 관계: {rels}개")

    # 가장 연결이 많은 엔티티 Top 10
    result = session.run("""
          MATCH (n)-[r]-()
          RETURN n.name AS name, labels(n) AS type, count(r) AS connections
          ORDER BY connections DESC
          LIMIT 10
      """)
    print("\n=== 허브 엔티티 (연결 많은 순) ===")
    for record in result:
        print(f"  {record['name']} ({record['type'][0]}): {record['connections']}개 연결")

노드: 571개, 관계: 409개

=== 허브 엔티티 (연결 많은 순) ===
  이란 (Country): 13개 연결
  이스라엘 (Country): 12개 연결
  엔씨소프트 (Organization): 10개 연결
  조희대 (Person): 8개 연결
  미국 (Country): 8개 연결
  남태현 (Person): 7개 연결
  모즈타바 하메네이 (Person): 7개 연결
  이재명 대통령 (Person): 7개 연결
  서서울미술관 (Organization): 7개 연결
  이재명 (Person): 6개 연결


In [14]:
with driver.session() as session:
    # "이란과 관련된 모든 엔티티는?"
    result = session.run("""
          MATCH (n)-[r]-(target {name: "이란"})
          RETURN n.name AS entity, type(r) AS relation
      """)
    print("=== 이란과 관련된 엔티티 ===")
    for record in result:
        print(f"  {record['entity']} - {record['relation']}")

    # "트럼프에서 2홉 이내 엔티티는?"
    result = session.run("""
          MATCH (start {name: "트럼프"})-[r1]-(mid)-[r2]-(end)
          RETURN DISTINCT end.name AS entity, type(r1) AS rel1, mid.name AS via, type(r2) AS rel2
          LIMIT 10
      """)
    print("\n=== 트럼프에서 2홉 이내 ===")
    for record in result:
        print(f"  트럼프 -{record['rel1']}-> {record['via']} -{record['rel2']}-> {record['entity']}")

=== 이란과 관련된 엔티티 ===
  이스라엘 - 타격
  아랍에미리트 - 드론_공격
  헤즈볼라 - 무장_정파
  이스라엘 - 합동_공격
  모즈타바 하메네이 - 최고지도자
  미국과 이스라엘 - 공동_공격
  핵무기 개발 - 부인
  미국과 이스라엘의 공격 보장 - 조건으로_제시하다
  이스라엘 - 공격
  미나브 여자 초등학교 - 소속
  미국 - 공습
  미국·이스라엘 연계 은행 - 경고
  이스라엘 - 공격

=== 트럼프에서 2홉 이내 ===
